In [3]:
%run ../setup/config

Box(children=(Label(value='Environment'), Dropdown(options=('dev', 'prod'), value='dev')))

Box(children=(Label(value='Cluster ID (vacío = serverless)'), Text(value='')))

In [4]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [5]:
spark.sql(f"""
ALTER TABLE {SILVER_MENU_FULL_TABLE}
SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5'
)
""")

print("Column mapping enabled on", SILVER_MENU_FULL_TABLE)

Column mapping enabled on dbr_dev.yanquiel_silver.slv_menu_items


In [6]:
spark.sql(f"""
ALTER TABLE {SILVER_MENU_FULL_TABLE}
RENAME COLUMN category TO menu_category
""")

display(spark.table(SILVER_MENU_FULL_TABLE).limit(5))

,menu_item_id,item_name,menu_category,price,_source_file,_ingestion_timestamp,effective_date,end_date,is_current
0,101,Hamburger,American,12.95,dbfs:/Volumes/dbr_dev/yanquiel_bronze/landing/menu/menu_items.csv,2026-08-07 00:08:28.494209,2026-08-07 10:11:30.878264,NaT,True
1,102,Cheeseburger,American,13.95,dbfs:/Volumes/dbr_dev/yanquiel_bronze/landing/menu/menu_items.csv,2026-08-07 00:08:28.494209,2026-08-07 10:11:30.878264,NaT,True
2,103,Hot Dog,American,9.00,dbfs:/Volumes/dbr_dev/yanquiel_bronze/landing/menu/menu_items.csv,2026-08-07 00:08:28.494209,2026-08-07 10:11:30.878264,NaT,True
3,104,Veggie Burger,American,10.50,dbfs:/Volumes/dbr_dev/yanquiel_bronze/landing/menu/menu_items.csv,2026-08-07 00:08:28.494209,2026-08-07 10:11:30.878264,NaT,True
4,105,Mac & Cheese,American,7.00,dbfs:/Volumes/dbr_dev/yanquiel_bronze/landing/menu/menu_items.csv,2026-08-07 00:08:28.494209,2026-08-07 10:11:30.878264,NaT,True


In [ ]:
# Example syntax — commented out to avoid accidental data loss during the demo
# spark.sql(f"""
# ALTER TABLE {SILVER_MENU_FULL_TABLE}
# DROP COLUMN _source_file
# """)

In [12]:
%md

## Data contracts :  why this matters here

With mergeSchema/autoMerge the table just adapts whenever the data's shape changes. It's convenient  no coordination needed but it also means the table's schema ends up being whatever the producer happens to send that day. New columns, type changes, renamed fields none of it gets reviewed, it just shows up.

A data contract is basically the opposite approach: producer and consumer agree on the schema ahead of time, and that agreement gets versioned and enforced (usually through a schema registry  Confluent already gives us one for this Kafka topic). The producer can't publish something that breaks the contract, so consumers don't have to double check the shape of every batch that comes in.

Trade off, roughly:

| | Schema evolution (mergeSchema) | Data contract |
|---|---|---|
| Iteration speed | Fast, no coordination | Slower, needs agreement between both sides |
| Risk to downstream consumers | Higher changes just flow through | Lower changes get reviewed first |
| When you find out about a change | After the fact | Before it ever lands |

This is basically what happened with the discount_code scenario in silver_orders.ipynb  the producer changed its output without telling anyone, and I only found out because the write to Silver failed. A data contract would've caught that at the source instead of letting it become a runtime error two layers downstream.


## Data contracts :  why this matters here

With mergeSchema/autoMerge the table just adapts whenever the data's shape changes. It's convenient  no coordination needed but it also means the table's schema ends up being whatever the producer happens to send that day. New columns, type changes, renamed fields none of it gets reviewed, it just shows up.

A data contract is basically the opposite approach: producer and consumer agree on the schema ahead of time, and that agreement gets versioned and enforced (usually through a schema registry  Confluent already gives us one for this Kafka topic). The producer can't publish something that breaks the contract, so consumers don't have to double check the shape of every batch that comes in.

Trade off, roughly:

| | Schema evolution (mergeSchema) | Data contract |
|---|---|---|
| Iteration speed | Fast, no coordination | Slower, needs agreement between both sides |
| Risk to downstream consumers | Higher changes just flow through | Lower changes get reviewed first |
| When you find out about a change | After the fact | Before it ever lands |

This is basically what happened with the discount_code scenario in silver_orders.ipynb  the producer changed its output without telling anyone, and I only found out because the write to Silver failed. A data contract would've caught that at the source instead of letting it become a runtime error two layers downstream.

In [13]:
spark.sql(f"OPTIMIZE {BRONZE_ORDER_FULL_TABLE}")

,path,metrics
0,,"{'numFilesAdded': 1, 'numFilesRemoved': 10, 'filesAdded': {'min': 15152, 'max': 15152, 'avg': 15152.0, 'totalFiles': 1, 'totalSize': 15152}, 'filesRemoved': {'min': 2560, 'max': 4745, 'avg': 3776.0, 'totalFiles': 10, 'totalSize': 37760}, 'partitionsOptimized': 0, 'zOrderStats': None, 'clusteringStats': None, 'numBins': 0, 'numBatches': 1, 'totalConsideredFiles': 10, 'totalFilesSkipped': 0, 'preserveInsertionOrder': True, 'numFilesSkippedToReduceWriteAmplification': 0, 'numBytesSkippedToReduceWriteAmplification': 0, 'startTimeMs': 1786218510405, 'endTimeMs': 1786218517519, 'totalClusterParallelism': 8, 'totalScheduledTasks': 1, 'autoCompactParallelismStats': None, 'deletionVectorStats': {'numDeletionVectorsRemoved': 0, 'numDeletionVectorRowsRemoved': 0}, 'recompressionCodec': None, 'numTableColumns': 6, 'numTableColumnsWithStats': 6, 'totalTaskExecutionTimeMs': 2591, 'skippedArchivedFiles': 0, 'clusteringMetrics': None, 'fileAgeHistogram': None}"


In [14]:
spark.sql(f"VACUUM {SILVER_MENU_FULL_TABLE} RETAIN 168 HOURS DRY RUN")

,path


In [16]:
spark.sql(f"""
ALTER TABLE {SILVER_ORDER_FULL_TABLE}
CLUSTER BY (order_id)
""")

spark.sql(f"OPTIMIZE {SILVER_ORDER_FULL_TABLE}")

,path,metrics
0,,"{'numFilesAdded': 0, 'numFilesRemoved': 0, 'filesAdded': {'min': None, 'max': None, 'avg': 0.0, 'totalFiles': 0, 'totalSize': 0}, 'filesRemoved': {'min': None, 'max': None, 'avg': 0.0, 'totalFiles': 0, 'totalSize': 0}, 'partitionsOptimized': 0, 'zOrderStats': None, 'clusteringStats': None, 'numBins': 0, 'numBatches': 0, 'totalConsideredFiles': 2, 'totalFilesSkipped': 0, 'preserveInsertionOrder': False, 'numFilesSkippedToReduceWriteAmplification': 0, 'numBytesSkippedToReduceWriteAmplification': 0, 'startTimeMs': 1786218690267, 'endTimeMs': 1786218695419, 'totalClusterParallelism': 8, 'totalScheduledTasks': 0, 'autoCompactParallelismStats': None, 'deletionVectorStats': {'numDeletionVectorsRemoved': 0, 'numDeletionVectorRowsRemoved': 0}, 'recompressionCodec': None, 'numTableColumns': 8, 'numTableColumnsWithStats': 8, 'totalTaskExecutionTimeMs': 0, 'skippedArchivedFiles': 0, 'clusteringMetrics': {'sizeOfTableInBytesBeforeLazyClustering': 11297, 'isNewMetadataCreated': True, 'isPOTriggered': False, 'isFull': False, 'approxClusteringQuality': None, 'approxClusteringQualityPerColumn': None, 'approxClusteringCoverage': None, 'compactionType': None, 'numFilesSkippedWithoutStats': 0, 'numFilesClassifiedToIntermediateNodes': 0, 'sizeOfFilesClassifiedToIntermediateNodesInBytes': 0, 'logicalSizeOfFilesClassifiedToIntermediateNodesInBytes': 0, 'numFilesClassifiedToLeafNodes': 2, 'sizeOfFilesClassifiedToLeafNodesInBytes': 11297, 'logicalSizeOfFilesClassifiedToLeafNodesInBytes': 11297, 'numThreadsForClassifier': None, 'clusterThresholdStrategy': 'log', 'minFileSize': 16777216, 'maxFileSize': 67108864, 'nodeMinNumFilesToCompact': 4, 'numIdealFiles': 0, 'numIdealFilesWithTrimmedStringMaxValue': 0, 'numAddedFilesWithSameMinMaxOnClusteringColumns': None, 'numClusteringTasksPlanned': 0, 'numClusteringTasksNotPlannedDueToPO': 0, 'totalSizeOfDataToClusterInBytesNotPlannedDueToPO': 0, 'numCompactionTasksPlanned': 0, 'numCompactionTasksPlannedUndoneDueToPO': 0, 'numOptimizeBatchesPlanned': 0, 'numLeafNodesExpanded': 0, 'numLeafNodesClustered': 0, 'numGetFilesForNodeCalls': 1, 'numSamplingJobs': 0, 'numLeafNodesCompacted': 0, 'numLeafNodesCompactedUndoneDueToPO': 0, 'numIntermediateNodesCompacted': 0, 'numIntermediateNodesCompactedUndoneDueToPO': 0, 'totalSizeOfDataToCompactInBytes': 0, 'totalSizeOfDataToCompactInBytesUndoneDueToPO': 0, 'totalLogicalSizeOfDataToCompactInBytes': 0, 'totalLogicalSizeOfDataToCompactInBytesUndoneDueToPO': 0, 'numIntermediateNodesClustered': 0, 'numFilesSkippedAfterExpansion': 0, 'numSmallFilesSkippedAfterExpansion': 0, 'totalSizeOfFilesSkippedAfterExpansionInBytes': 0, 'totalSizeOfSmallFilesSkippedAfterExpansionInBytes': 0, 'totalLogicalSizeOfFilesSkippedAfterExpansionInBytes': 0, 'totalLogicalSizeOfSmallFilesSkippedAfterExpansionInBytes': 0, 'totalSizeOfDataToRewriteInBytes': 0, 'totalLogicalSizeOfDataToRewriteInBytes': 0, 'timeMetrics': {'classifierTimeMs': 194, 'optimizerTimeMs': 402, 'metadataLoadTimeMs': 0, 'totalGetFilesForNodeCallsTimeMs': 257, 'totalSamplingTimeMs': 0, 'metadataCreationTimeMs': 1530}, 'maxOptimizeBatchesInParallel': 2, 'currentIteration': 1, 'maxIterations': 5, 'clusteringStrategy': 'sizeAware', 'splitStrategy': 'defaultSplitStrategy', 'numDowncastDimensions': None, 'withPredicates': False, 'numFilesSkippedByPredicates': 0, 'predicates': None, 'isPipelinedOptimizer': False, 'pipelinedTotalPlanningTimeMs': 0, 'pipelinedTotalBatchWaitTimeMs': 0, 'pipelinedPostFirstBatchWaitTimeMs': 0, 'coordinatorEventMetrics': None, 'pipelinedOptimizerStatus': None, 'pipelinedAssemblerMetrics': None}, 'fileAgeHistogram': None}"
1,,"{'numFilesAdded': 1, 'numFilesRemoved': 2, 'filesAdded': {'min': 9703.0, 'max': 9703.0, 'avg': 9703.0, 'totalFiles': 1, 'totalSize': 9703}, 'filesRemoved': {'min': 2754.0, 'max': 8543.0, 'avg': 5648.5, 'totalFiles': 2, 'totalSize': 11297}, 'partitionsOptimized': 0, 'zOrderStats': None, 'clusteringStats': None, 'numBins': 0, 'numBatches': 1, 'totalConsideredFiles': 2, 'to

In [17]:
spark.sql(f"""
ALTER TABLE {SILVER_MENU_FULL_TABLE}
ADD CONSTRAINT positive_price CHECK (price > 0)
""")

spark.sql(f"""
ALTER TABLE {SILVER_MENU_FULL_TABLE}
ADD CONSTRAINT valid_menu_item_id CHECK (menu_item_id IS NOT NULL)
""")

""


In [19]:
df_bad_row = spark.createDataFrame(
    [(999, "Test Item", "Test", -5.0, "manual_test", True)],
    ["menu_item_id", "item_name", "menu_category", "price", "_source_file", "is_current"]
).withColumn("end_date", F.lit(None).cast("timestamp")) \
 .withColumn("effective_date", F.current_timestamp()) \
 .withColumn("_ingestion_timestamp", F.current_timestamp())

try:
    df_bad_row.write.format("delta").mode("append").saveAsTable(SILVER_MENU_FULL_TABLE)
except Exception as e:
    print(f"Data quality constraint rejected the write as expected:\n{e}")

Data quality constraint rejected the write as expected:
(com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException) [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint positive_price (price > 0) violated by row with values:
 - price : -5.0.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException
	at com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:82)
	at com.databricks.photon.PhotonException$.getSparkException(PhotonException.scala:466)
	at com.databricks.photon.PhotonWriteResultHandler.getResult(PhotonWriteStageExec.scala:153)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.open(PhotonBasicEvaluatorFactory.scala:264)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNextImpl(PhotonBasicEvaluatorFactory.scala:269)
	at com.databricks.photon.PhotonBasicE